# Multi-source, question-focused summarizer — **Google ADK**

This is one of four notebooks that build the **same** small agentic app, each in a different
framework, so you can compare how they wire up identical ideas. Here it's **Google ADK**.

**The task.** You give the agent either a **list of URLs** *or* a **local file path**, plus a
**question**. The agent **decides on its own** how to fetch, parse, and summarize — then returns a
summary focused on your question.

```
            ┌─────────────────────────── the agent decides ───────────────────────────┐
 question + │  URL?  ──▶  fetch_url  (scraper, an MCP server)  ──▶  parse_html (tool)   │
 sources ──▶│  .pdf? ──▶  parse_pdf  (tool)                                             │──▶ summarize (tool) ──▶ answer
            └──────────────────────────────────────────────────────────────────────────┘
```

**What is shared vs. what differs.** The parsers and the summarizer are plain Python tools imported
from `src/` — *identical* in every notebook. The **scraper is exposed as an MCP server** (a separate
process), so each framework has to show how it *consumes* MCP. The only thing that changes between
notebooks is the framework-specific wiring below: how it builds the model, registers tools, connects
the MCP server, runs the agent, and handles memory. Read those side by side.

## 1. Install

Everything is declared in `pyproject.toml`; install the project (editable) plus dev tools. If you
already ran `uv pip install -e ".[dev]"` in your venv, this is a no-op.

To use a **local** model instead of OpenAI, start Ollama first: `ollama serve` and
`ollama pull qwen3.5:4b` (see the project README).

In [ ]:
%pip install -q -e ".[dev]"

## 2. Choose the model provider

One env var flips the whole notebook between **OpenAI** (`gpt-5-mini`) and a **local Ollama** model
(`qwen3.5:4b`, served on an OpenAI-compatible endpoint). `load_llm_config()` reads it and returns a
small config object that both the summarizer tool and the agent below are built from.

In [ ]:
import os

from dotenv import load_dotenv

from src.config import load_llm_config

load_dotenv()  # reads .env (OPENAI_API_KEY, etc.)

# Flip this to "ollama" to run fully locally (requires `ollama serve` + the pulled model).
os.environ["LLM_PROVIDER"] = "openai"

cfg = load_llm_config()  # reads LLM_PROVIDER at call time, so the line above takes effect
print(f"provider={cfg.provider!r}  model={cfg.model!r}  base_url={cfg.base_url!r}")

## 3. The shared tools (and why the scraper is an MCP server)

Three of the four tools are ordinary in-process Python functions, imported from `src/` and reused
**unchanged** by every framework:

- `html_to_text(html)` — strip boilerplate from raw HTML and return clean text.
- `pdf_to_text(path)` — extract clean text from a local PDF.
- `summarize_for_question(text, question)` — a question-focused summary (LLM call, works for both providers).

The **scraper** is different: it runs as a **separate process** and is reached over the **Model
Context Protocol (MCP)** via stdio. This is the realistic shape for an external capability — a tool
the agent talks to over a protocol rather than calling in-process. Each framework section shows how
it launches and consumes that server (the spawn command is the same everywhere).

In [ ]:
import os
import sys

from src.tools.parsers.html_parser import html_to_text
from src.tools.parsers.pdf_parser import pdf_to_text
from src.tools.summarizer import summarize_for_question

# How every framework spawns the scraper MCP server (stdio subprocess):
REPO_ROOT = os.getcwd()  # notebooks run from the repo root
SCRAPER_CMD = sys.executable  # this venv's python
SCRAPER_ARGS = ["-m", "src.mcp_servers.scraper_server"]
print("scraper launch:", SCRAPER_CMD, *SCRAPER_ARGS)

# Example inputs reused by the run cells below (swap in your own URL / question / PDF):
EXAMPLE_URL = "https://en.wikipedia.org/wiki/Large_language_model"
EXAMPLE_Q = "What are the main capabilities and risks of large language models?"

## 4. Sanity check the MCP server

Before wiring it into an agent, confirm the server starts and advertises its tools. This spawns the
server over stdio and lists what the agent will receive (`fetch_url`, `fetch_urls`). You can also run
it standalone in a terminal: `python -m src.mcp_servers.scraper_server`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def list_scraper_tools() -> list[str]:
    params = StdioServerParameters(
        command=SCRAPER_CMD, args=SCRAPER_ARGS, cwd=REPO_ROOT, env=dict(os.environ)
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [t.name for t in tools.tools]


await list_scraper_tools()

## 5. Google ADK

ADK is code-first around `LlmAgent` + a `Runner` with a session service. Tools are plain typed
functions (the docstring + hints become the schema). The MCP scraper is an `MCPToolset` placed
directly in `tools`. ADK is Gemini-first, so to use OpenAI/Ollama we wrap the model in `LiteLlm`
(`openai/<model>`; for Ollama we also pass `api_base`/`api_key`).

Memory is **session state**: we seed a `routing_hint` into the session and inject it into the agent's
instruction with a `{routing_hint}` placeholder.

**Build the model (via LiteLlm), the plain-function tools, and the MCP toolset.**

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import MCPToolset, StdioConnectionParams
from google.genai import types

from src.config import litellm_model_string


def build_model(cfg):
    if cfg.provider == "ollama":
        return LiteLlm(model=litellm_model_string(cfg), api_base=cfg.base_url, api_key=cfg.api_key)
    return LiteLlm(model=litellm_model_string(cfg), api_key=cfg.api_key)


def parse_html(html: str) -> str:
    """Convert raw HTML (from fetch_url) into clean text."""
    return html_to_text(html)


def parse_pdf(path: str) -> str:
    """Extract clean text from a local PDF path."""
    return pdf_to_text(path)


def summarize(text: str, question: str) -> str:
    """Return a question-focused summary of text."""
    return summarize_for_question(text, question)


# The scraper MCP server, as a toolset (stdio subprocess):
scraper = MCPToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=SCRAPER_CMD, args=SCRAPER_ARGS, cwd=REPO_ROOT, env=dict(os.environ)
        ),
        timeout=30.0,
    )
)

In [ ]:
INSTRUCTION = (
    "Routing hint: {routing_hint}\n"
    "You answer a question by summarizing sources. For a local .pdf path call parse_pdf; "
    "for a URL call fetch_url then parse_html; then call summarize. Stay grounded in the content."
)
agent = LlmAgent(
    name="summarizer",
    model=build_model(cfg),
    instruction=INSTRUCTION,
    tools=[parse_html, parse_pdf, summarize, scraper],
)

session_service = InMemorySessionService()
runner = Runner(agent=agent, app_name="summarizer", session_service=session_service)

HINT = "for .pdf use parse_pdf; for URLs fetch_url then parse_html; stay grounded."


async def answer(sources: str, question: str, session_id: str = "s1") -> str:
    await session_service.create_session(
        app_name="summarizer", user_id="u1", session_id=session_id, state={"routing_hint": HINT}
    )  # seed the persistent hint into state
    prompt = f"Sources: {sources}\nQuestion: {question}"
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    final = ""
    async for ev in runner.run_async(user_id="u1", session_id=session_id, new_message=msg):
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    return final


# Example A — a URL: expect fetch_url -> parse_html -> summarize
print(await answer(EXAMPLE_URL, EXAMPLE_Q, session_id="s1"))

In [ ]:
# Example B — a local file: expect parse_pdf -> summarize (new session id)
if os.path.exists("data/sample.pdf"):
    print(await answer("data/sample.pdf", "What is the main conclusion?", session_id="s2"))
else:
    print("Put a PDF at data/sample.pdf to try the local-file path.")

**Memory.** ADK keeps memory in **session state**. We pass `state={"routing_hint": ...}` into
`create_session(...)`, and `LlmAgent.instruction` substitutes `{routing_hint}` from that state at run
time — a clean way to inject a persisted hint. `InMemorySessionService` is per-process; for storage
that survives restarts use `DatabaseSessionService` (needs `sqlalchemy` / `google-adk[db]`).

## 6. Step back: the same app, four wirings

You've now seen this framework end to end. Compared with the others:

- **Pydantic AI** — typed and compact; tools are decorated functions, the MCP server is a `toolset`,
  multi-turn is explicit (`message_history`). No built-in persistent store.
- **LangGraph** — a graph with a prebuilt ReAct loop; MCP tools arrive via an adapter; real
  persistence through a `checkpointer` + `thread_id`.
- **OpenAI Agents SDK** — the leanest surface; native `mcp_servers=[...]`; the cleanest *persistent*
  memory via `SQLiteSession` (writes to disk).
- **Google ADK** — code-first around `LlmAgent` + `Runner`; MCP via `MCPToolset`; memory through
  session state, with a `{routing_hint}` injected into the instruction.

Same parsers, same summarizer, same MCP scraper — only the orchestration differs. That difference is
the thing this lesson is about.